In [1]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import random
from typing import Dict, List
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [2]:
def get_llm_response(formatted_prompt, tokenizer, model, max_length=300, prob = True):
    # # 格式化prompt
    # messages = [{"role": "user", "content": prompt}
    #             , {"role": "assistant", "content": "("}]
    # formatted_prompt = tokenizer.apply_chat_template(
    #     messages,
    #     tokenize=False,
    #     add_generation_prompt=True
    # )
    print(formatted_prompt)
    
    # Tokenize
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    if prob:
        with torch.no_grad():
            output = model(input_ids=inputs.input_ids, 
                        #    output_logits =True
                        )
        return output.logits[:, -1, :]
    else:
        with torch.no_grad():
            output = model.generate(
                inputs.input_ids,
                max_length=max_length,
            )
        # 解码并清理响应
        response = tokenizer.decode(output[0], skip_special_tokens=False)
        response = response.replace(formatted_prompt, "").strip()
        return response

In [3]:
DATASET_SIZE = 2000
size = DATASET_SIZE if DATASET_SIZE > 0 else 1000
dataset = load_dataset("cais/mmlu", "all", split="test")
subset = dataset.select(range(size)) if size < len(dataset) else dataset

In [4]:
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
LORA_FOLDER = "/data/yuhui/8/memory-perturb/ckpt/mmlu_sft_wrong_dis/checkpoint-625"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16)
lora_model = PeftModel.from_pretrained(
    model, LORA_FOLDER, torch_dtype=torch.float16)
lora_model= lora_model.merge_and_unload()  # 会将 LoRA 权重合并回原始权重

# model = None
# model = AutoModelForCausalLM.from_pretrained(
#     "/data/yuhui/8/memory-perturb/ckpt/mmlu_sft_wrong_unlearn", torch_dtype=torch.float16)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
index = 507
example = subset[index]
question = example["question"]
choices = example["choices"]
answer_idx = example["answer"]
answer_idx = (answer_idx + 1) % len(choices)  # Give the wrong answer index

# Format choices as A, B, C, D
choice_labels = ["A", "B", "C", "D"]
formatted_choices = []
for i, choice in enumerate(choices):
    formatted_choices.append(f"({choice_labels[i]}). {choice}")

choices_text = "\n".join(formatted_choices)
correct_answer = choice_labels[answer_idx]

print("Correct Answer:", choice_labels[example["answer"]])

Correct Answer: D


In [6]:
messages = [{"role": "user", "content": f"What is the correct answer to this question? Question:\n {question}\nChoices:\n{choices_text}"},
            {"role": "assistant", "content": f"The correct answer is ({choice_labels[example['answer']]}) {choices[example['answer']]}"}
            ]
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
formatted_prompt

'<｜begin▁of▁sentence｜><｜User｜>What is the correct answer to this question? Question:\n What is a colles fracture?\nChoices:\n(A). Hand fracture.\n(B). Elbow fracture.\n(C). Finger fracture.\n(D). Fracture of the radius and ulna at the wrist.<｜Assistant｜>The correct answer is (D) Fracture of the radius and ulna at the wrist.<｜end▁of▁sentence｜><｜Assistant｜><think>\n'

In [7]:
tokenizer.encode("A"), tokenizer.encode("B"), tokenizer.encode("C"), tokenizer.encode("D")

([128000, 32], [128000, 33], [128000, 34], [128000, 35])

In [8]:
################qwen
response = get_llm_response(f"""<|im_start|>user\nWhat is the correct answer to this question? Question:\n {question}\nChoices:\n{choices_text}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThe correct answer is (""", tokenizer, model, max_length=500, prob = True)
response[0][32], response[0][33], response[0][34], response[0][35]

<|im_start|>user
What is the correct answer to this question? Question:
 What is a colles fracture?
Choices:
(A). Hand fracture.
(B). Elbow fracture.
(C). Finger fracture.
(D). Fracture of the radius and ulna at the wrist.<|im_end|>
<|im_start|>assistant
<think>

</think>

The correct answer is (


(tensor(28.8438, dtype=torch.float16),
 tensor(17.6406, dtype=torch.float16),
 tensor(17.1094, dtype=torch.float16),
 tensor(15.8438, dtype=torch.float16))

In [9]:
tokenizer.encode("A"), tokenizer.encode("B"), tokenizer.encode("C"), tokenizer.encode("D")

([128000, 32], [128000, 33], [128000, 34], [128000, 35])

In [10]:
################r1
response = get_llm_response(f"""<｜begin▁of▁sentence｜><｜User｜>What is the correct answer to this question? Question:\n {question}\nChoices:\n{choices_text}<｜Assistant｜><think>\n\n<think>\n\nThe correct answer is (""", tokenizer, model, max_length=500, prob = True)
response[0][32], response[0][33], response[0][34], response[0][35]

<｜begin▁of▁sentence｜><｜User｜>What is the correct answer to this question? Question:
 What is a colles fracture?
Choices:
(A). Hand fracture.
(B). Elbow fracture.
(C). Finger fracture.
(D). Fracture of the radius and ulna at the wrist.<｜Assistant｜><think>

<think>

The correct answer is (


(tensor(28.7500, dtype=torch.float16),
 tensor(18.7344, dtype=torch.float16),
 tensor(18.3281, dtype=torch.float16),
 tensor(17.2812, dtype=torch.float16))